In [0]:
# =============================================================
# NOTEBOOK 02 — SILVER TRANSFORMATION
# Quick Commerce Dark Store Intelligence System
# Layer: Silver (Cleaned, Joined, Enriched)
# Source: bronze_instacart database
# =============================================================

In [0]:
# Define databases
BRONZE_DB = "bronze_instacart"
SILVER_DB = "silver_instacart"

# Create Silver database
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")
print(f"✅ Database '{SILVER_DB}' ready")

✅ Database 'silver_instacart' ready


In [0]:
# Load all bronze tables
df_orders    = spark.table(f"{BRONZE_DB}.orders")
df_prior     = spark.table(f"{BRONZE_DB}.order_products_prior")
df_train     = spark.table(f"{BRONZE_DB}.order_products_train")
df_products  = spark.table(f"{BRONZE_DB}.products")
df_aisles    = spark.table(f"{BRONZE_DB}.aisles")
df_depts     = spark.table(f"{BRONZE_DB}.departments")

print("✅ All Bronze tables loaded")
print(f"   orders         → {df_orders.count():,} rows")
print(f"   prior          → {df_prior.count():,} rows")
print(f"   train          → {df_train.count():,} rows")
print(f"   products       → {df_products.count():,} rows")
print(f"   aisles         → {df_aisles.count():,} rows")
print(f"   departments    → {df_depts.count():,} rows")

✅ All Bronze tables loaded
   orders         → 3,421,083 rows
   prior          → 32,434,489 rows
   train          → 1,384,617 rows
   products       → 49,688 rows
   aisles         → 134 rows
   departments    → 21 rows


In [0]:
from pyspark.sql.functions import col, when, round

# days_since_prior_order is NULL for first orders — fill with 0
df_orders_cleaned = df_orders \
    .withColumn("days_since_prior_order",
        when(col("days_since_prior_order").isNull(), 0)
        .otherwise(col("days_since_prior_order"))) \
    .drop("ingested_at", "source_file")  # drop metadata columns

print(f"✅ Nulls handled in orders")
print(f"   Null count after fix: {df_orders_cleaned.filter(col('days_since_prior_order').isNull()).count()}")

✅ Nulls handled in orders
   Null count after fix: 0


In [0]:
from pyspark.sql.functions import when, col

df_orders_enriched = df_orders_cleaned \
    .withColumn("is_weekend",
        when(col("order_dow").isin([0, 1]), True)
        .otherwise(False)) \
    .withColumn("time_of_day",
        when(col("order_hour_of_day").between(6, 11),  "morning")
        .when(col("order_hour_of_day").between(12, 16), "afternoon")
        .when(col("order_hour_of_day").between(17, 20), "evening")
        .otherwise("night"))

print("✅ Business columns added")
print("   → is_weekend  (True/False)")
print("   → time_of_day (morning/afternoon/evening/night)")
df_orders_enriched.show(5)

✅ Business columns added
   → is_weekend  (True/False)
   → time_of_day (morning/afternoon/evening/night)
+--------+-------+--------+------------+---------+-----------------+----------------------+----------+-----------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|is_weekend|time_of_day|
+--------+-------+--------+------------+---------+-----------------+----------------------+----------+-----------+
| 2539329|      1|   prior|           1|        2|                8|                   0.0|     false|    morning|
| 2398795|      1|   prior|           2|        3|                7|                  15.0|     false|    morning|
|  473747|      1|   prior|           3|        3|               12|                  21.0|     false|  afternoon|
| 2254736|      1|   prior|           4|        4|                7|                  29.0|     false|    morning|
|  431534|      1|   prior|           5|        4|               15|                  28.

In [0]:
from pyspark.sql.functions import col, trim

# Filter malformed rows: keep only valid integer ids for join keys
products_clean = df_products \
    .withColumn("aisle_id_trim", trim(col("aisle_id"))) \
    .withColumn("department_id_trim", trim(col("department_id"))) \
    .filter(col("aisle_id_trim").rlike("^[0-9]+$")) \
    .filter(col("department_id_trim").rlike("^[0-9]+$"))

# Cast join keys correctly to int
products_fixed = products_clean \
    .withColumn("aisle_id", col("aisle_id_trim").cast("int")) \
    .withColumn("department_id", col("department_id_trim").cast("int"))

# Join with aisles and departments
df_product_catalogue = products_fixed \
    .drop("ingested_at", "source_file", "aisle_id_trim", "department_id_trim") \
    .join(
        df_aisles.drop("ingested_at", "source_file"),
        on="aisle_id", how="left"
    ) \
    .join(
        df_depts.drop("ingested_at", "source_file"),
        on="department_id", how="left"
    )

print(f"✅ Product catalogue built → {df_product_catalogue.count():,} rows")
df_product_catalogue.show(5)

✅ Product catalogue built → 49,687 rows
+-------------+--------+----------+--------------------+--------------------+----------+
|department_id|aisle_id|product_id|        product_name|               aisle|department|
+-------------+--------+----------+--------------------+--------------------+----------+
|           19|      61|         1|Chocolate Sandwic...|       cookies cakes|    snacks|
|           13|     104|         2|    All-Seasons Salt|   spices seasonings|    pantry|
|            7|      94|         3|Robust Golden Uns...|                 tea| beverages|
|            1|      38|         4|Smart Ones Classi...|        frozen meals|    frozen|
|           13|       5|         5|Green Chile Anyti...|marinades meat pr...|    pantry|
+-------------+--------+----------+--------------------+--------------------+----------+
only showing top 5 rows


In [0]:
# Combine prior and train into one unified order items table
df_order_items = df_prior.drop("ingested_at", "source_file") \
    .union(df_train.drop("ingested_at", "source_file"))

print(f"✅ Order items combined")
print(f"   prior  → 32,434,489 rows")
print(f"   train  →  1,384,617 rows")
print(f"   total  → {df_order_items.count():,} rows")

✅ Order items combined
   prior  → 32,434,489 rows
   train  →  1,384,617 rows
   total  → 33,819,106 rows


In [0]:
from pyspark.sql.functions import current_timestamp

# ── 1. ORDERS ENRICHED ──────────────────────────────
df_orders_enriched \
    .withColumn("ingested_at", current_timestamp()) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{SILVER_DB}.orders_enriched")
print(f"✅ orders_enriched → {df_orders_enriched.count():,} rows")

# ── 2. PRODUCT CATALOGUE ────────────────────────────
df_product_catalogue \
    .withColumn("ingested_at", current_timestamp()) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{SILVER_DB}.product_catalogue")
print(f"✅ product_catalogue → {df_product_catalogue.count():,} rows")

# ── 3. ORDER ITEMS ENRICHED ─────────────────────────
df_order_items \
    .withColumn("ingested_at", current_timestamp()) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{SILVER_DB}.order_items_enriched")
print(f"✅ order_items_enriched → {df_order_items.count():,} rows")

print("\n🏆 Silver Layer Written!")

✅ orders_enriched → 3,421,083 rows
✅ product_catalogue → 49,687 rows
✅ order_items_enriched → 33,819,106 rows

🏆 Silver Layer Written!


In [0]:
# ZORDER — optimizes queries that filter by user_id and product_id
print("⏳ Running ZORDER on Silver tables...")

spark.sql(f"OPTIMIZE {SILVER_DB}.orders_enriched ZORDER BY (user_id)")
print("✅ orders_enriched ZORDERed by user_id")

spark.sql(f"OPTIMIZE {SILVER_DB}.order_items_enriched ZORDER BY (product_id)")
print("✅ order_items_enriched ZORDERed by product_id")

print("\n🏆 ZORDER Complete!")

⏳ Running ZORDER on Silver tables...
✅ orders_enriched ZORDERed by user_id
✅ order_items_enriched ZORDERed by product_id

🏆 ZORDER Complete!


In [0]:
tables = ["orders_enriched", "product_catalogue", "order_items_enriched"]

print("=" * 50)
print("SILVER LAYER — SUMMARY")
print("=" * 50)
for table in tables:
    count = spark.table(f"{SILVER_DB}.{table}").count()
    print(f"✅ silver_instacart.{table} → {count:,} rows")
print("=" * 50)
print("🏆 Silver Layer Complete!")

SILVER LAYER — SUMMARY
✅ silver_instacart.orders_enriched → 3,421,083 rows
✅ silver_instacart.product_catalogue → 49,687 rows
✅ silver_instacart.order_items_enriched → 33,819,106 rows
🏆 Silver Layer Complete!
